In [345]:
# https://platform.olimpiada-ai.ro/ro/problems/198

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

In [346]:
train = pd.read_csv("/kaggle/input/datasets/abukanabek/robots-in-the-maze/train.csv")
test = pd.read_csv("/kaggle/input/datasets/abukanabek/robots-in-the-maze/test.csv")

train.shape, test.shape

((1200, 25), (400, 24))

In [347]:
train.head()

,robot_id,arena_type,weather,difficulty,laps,avg_speed_mps,distance_m,battery_used_pct,collisions,unique_cells,...,efficiency_score,coverage_ratio,scan_turn_sync,pickup_efficiency,detour_index,safety_margin,patrol_consistency,speed_burst,risk_index,strategy_label
0,TR_0000,urban,clear,2,3,1.561222,2999.630719,72.252973,7,247.530750,...,13.290300,6.719937,33.092663,4.839689,-10.000000,10.983818,-0.176148,9.821879,7.942570,explorer
1,TR_0001,urban,clear,1,5,1.442525,3154.068573,64.676535,6,334.130274,...,26.398660,5.192224,30.436153,1.095235,-10.000000,9.600871,3.270004,9.372641,3.101063,explorer
2,TR_0002,forest,fog,3,2,1.722279,3065.318986,80.265380,12,259.159918,...,10.000000,3.762810,18.799661,0.986697,-5.754699,0.000000,-8.351089,14.195603,13.897243,sprinter
3,TR_0003,forest,fog,4,4,1.603709,3237.298212,76.131047,11,247.686196,...,10.000000,2.974647,21.146474,1.197847,-8.658303,4.695742,-0.860512,13.064054,9.994055,sprinter
4,TR_0004,urban,clear,2,4,1.806128,3730.035162,72.061673,4,279.860091,...,33.829585,6.206864,32.408799,4.544147,-7.942540,11.964464,4.821596,9.304336,1.649628,explorer


In [348]:
train['strategy_label'].value_counts()

strategy_label
explorer     352
collector    307
sprinter     304
guardian     237
Name: count, dtype: int64

In [349]:
n_arenas = train['arena_type'].nunique()
max_speed = train['avg_speed_mps'].max()
most_freq_arena = train['arena_type'].value_counts().sort_values(ascending=False).index[0]
max_collection = train['items_collected'].max()

n_arenas, max_speed, most_freq_arena, max_collection

(4, 2.641200394995685, 'forest', 22)

In [350]:
from sklearn.model_selection import train_test_split

features = [c for c in train.columns if c not in ['robot_id', 'strategy_label']]
cat_features = [c for c in train.select_dtypes('object').columns if c in features]
target_col = 'strategy_label'

X, y = train[features], train[target_col]
X_test = test[features]

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.4, stratify=y, random_state=42)
X_train.shape, X_valid.shape

((720, 23), (480, 23))

In [351]:
from catboost import Pool

train_pool = Pool(X_train, y_train, cat_features=cat_features)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features)

In [352]:
from catboost import CatBoostClassifier

params = {
    'iterations': 160,
    'loss_function': 'MultiClass',
    'eval_metric': 'TotalF1:average=Macro',
    'metric_period': 20,
    'max_depth': 2,
    'learning_rate': 0.15,
    'random_state': 42
}

model = CatBoostClassifier(**params)

model.fit(train_pool, eval_set=valid_pool)

0:	learn: 0.5410544	test: 0.5086621	best: 0.5086621 (0)	total: 3.57ms	remaining: 568ms
20:	learn: 0.9038621	test: 0.8756452	best: 0.8756452 (20)	total: 34.7ms	remaining: 230ms
40:	learn: 0.9358656	test: 0.8945222	best: 0.8945222 (40)	total: 63.4ms	remaining: 184ms
60:	learn: 0.9478098	test: 0.9006140	best: 0.9006140 (60)	total: 92.1ms	remaining: 149ms
80:	learn: 0.9507264	test: 0.9071944	best: 0.9071944 (80)	total: 121ms	remaining: 118ms
100:	learn: 0.9599357	test: 0.9130973	best: 0.9130973 (100)	total: 150ms	remaining: 87.3ms
120:	learn: 0.9596872	test: 0.9188015	best: 0.9188015 (120)	total: 178ms	remaining: 57.3ms
140:	learn: 0.9649190	test: 0.9166412	best: 0.9188015 (120)	total: 208ms	remaining: 28ms
159:	learn: 0.9661779	test: 0.9168758	best: 0.9188015 (120)	total: 235ms	remaining: 0us

bestTest = 0.9188014963
bestIteration = 120

Shrink model to first 121 iterations.


In [353]:
y_pred = model.predict(X_test).flatten().tolist()
y_pred[:5]

['guardian', 'guardian', 'explorer', 'sprinter', 'explorer']

In [354]:
subm = pd.DataFrame({
    'robot_id': ['GLOBAL'] * 4 + test['robot_id'].tolist(),
    'subtaskID': [1, 2, 3, 4] + [5] * len(test),
    'answer': [n_arenas, max_speed, most_freq_arena, max_collection] + y_pred
})

subm.to_csv("submission.csv", index=False)
subm

,robot_id,subtaskID,answer
0,GLOBAL,1,4
1,GLOBAL,2,2.6412
2,GLOBAL,3,forest
3,GLOBAL,4,22
4,TS_0000,5,guardian
...,...,...,...
399,TS_0395,5,explorer
400,TS_0396,5,explorer
401,TS_0397,5,collector
402,TS_0398,5,collector
